# CSI800 ML Mainline Family V3 Full Families

这份 notebook 补齐 v2 没有测全的三块主线：

1. `rank_industry_lgb`：行业内相对排名特征，不再只看 raw factor。
2. `recall_rerank_lgb`：第一阶段多源召回，第二阶段单独 reranker 重排。
3. `sleeve_portfolio_family`：固定权重组合层，不用 2025+ 调权重。

Baseline 仍然只是 anchor：`raw_v22a_lgb_alpha_vs_alla`。

统一协议：CSI800 月频、label-safe 训练、2025+ OOS 只做报告，不反向调参。


In [ ]:
from jqdata import *
from jqfactor import get_factor_values

import datetime
import gc
import hashlib
import os
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 240)


## 1. Config

In [ ]:
UNIVERSE_INDEX = "000906.XSHG"
BENCHMARK_CSI800 = "000906.XSHG"
BENCHMARK_ALLA = "000985.XSHG"

DATA_START = "2019-01-01"
DATA_END_FOR_LABEL = "2026-05-31"
TRAIN_START = "2019-01-01"
TRAIN_END = "2024-12-31"
VALID_START = "2025-01-01"
VALID_END = "2026-04-28"

MIN_LISTING_DAYS = 180
FACTOR_BATCH_SIZE = 10
PRICE_BATCH_SIZE = 160
LGB_NUM_THREADS = 2

STOCK_NUM = 10
TOP_N_CANDIDATES = 30
GROUP_RECALL_N = 40
RERANK_RECALL_N = 80
MAX_PER_INDUSTRY = 2

CACHE_DIR = "csi800_ml_mainline_cache"
OUT_DIR = "csi800_ml_mainline_family_v3_outputs"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

USE_CACHE = True
REBUILD_DATASET = False

LGB_PROFILE = {
    "num_boost_round": 120,
    "params": {
        "learning_rate": 0.025,
        "num_leaves": 11,
        "max_depth": 3,
        "min_data_in_leaf": 700,
        "feature_fraction": 0.70,
        "bagging_fraction": 0.75,
        "bagging_freq": 1,
        "lambda_l1": 1.0,
        "lambda_l2": 5.0,
        "min_gain_to_split": 0.00001,
    },
}

print("mainline family v3 full families")
print("baseline anchor = raw v22a_37 + alpha_vs_alla + fixed_regularized_120")
print("challengers = rank_industry / true_recall_rerank / sleeve_portfolio_family")
print("train:", TRAIN_START, TRAIN_END, "valid:", VALID_START, VALID_END)


## 2. Factor Menus And Family Definitions

In [ ]:
FACTOR_SET_V22A_37 = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit", "ACCA",
    "growth", "net_working_capital", "operating_profit_per_share",
    "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    "super_quick_ratio", "MLEV", "debt_to_equity_ratio", "debt_to_tangible_equity_ratio",
    "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "liquidity", "beta",
    "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "Skewness20", "Kurtosis20",
]

FACTOR_GROUPS = {
    "value_cashflow": [
        "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
        "cash_flow_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
        "adjusted_profit_to_total_profit", "net_operate_cash_flow_per_share",
        "operating_profit_per_share", "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "growth", "net_working_capital", "super_quick_ratio", "MLEV",
        "debt_to_equity_ratio", "debt_to_tangible_equity_ratio", "ACCA",
    ],
    "momentum_risk": [
        "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
        "beta", "Skewness20", "Kurtosis20",
    ],
    "technical_volume": [
        "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "liquidity",
    ],
}

MANUAL_SCORE_CONFIG = [
    ("Rank1M", 1, 0.18),
    ("momentum", 1, 0.14),
    ("beta", 1, 0.08),
    ("growth", 1, 0.08),
    ("roe_ttm", 1, 0.06),
    ("cash_flow_to_price_ratio", 1, 0.05),
    ("earnings_yield", 1, 0.04),
    ("book_to_price_ratio", 1, 0.03),
    ("Variance20", 1, 0.03),
    ("VOL10", 1, 0.02),
]

RISK_PENALTY_FACTORS = ["beta", "Variance20", "VOL10", "liquidity"]

HYBRID_LIGHT_RAW_COLS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio", "sales_to_price_ratio",
    "roe_ttm", "roa_ttm", "gross_profit_ttm", "growth", "net_working_capital",
    "momentum", "Rank1M", "sharpe_ratio_60", "beta", "Variance20", "VOL10", "VMACD", "VOSC", "liquidity",
]
HYBRID_LIGHT_TRANSFORM_COLS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio", "roe_ttm",
    "growth", "momentum", "Rank1M", "sharpe_ratio_60", "beta", "Variance20", "VOL10", "liquidity",
]


def unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

all_factors = []
all_factors.extend(FACTOR_SET_V22A_37)
for k in FACTOR_GROUPS:
    all_factors.extend(FACTOR_GROUPS[k])
for item in MANUAL_SCORE_CONFIG:
    all_factors.append(item[0])
all_factors.extend(RISK_PENALTY_FACTORS)
all_factors.extend(HYBRID_LIGHT_RAW_COLS)
all_factors.extend(HYBRID_LIGHT_TRANSFORM_COLS)
ALL_FACTORS = unique_keep_order(all_factors)

print("v22a factors:", len(FACTOR_SET_V22A_37))
print("factor groups:", dict((k, len(v)) for k, v in FACTOR_GROUPS.items()))
print("hybrid raw/transform:", len(HYBRID_LIGHT_RAW_COLS), len(HYBRID_LIGHT_TRANSFORM_COLS))
print("union factors:", len(ALL_FACTORS))


## 3. Data Builders

In [ ]:
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def downcast_numeric_df(df):
    if df is None or df.empty:
        return df
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if np.issubdtype(dtype, np.floating):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif np.issubdtype(dtype, np.integer):
            out[col] = pd.to_numeric(out[col], downcast="integer")
    return out

def cache_key(parts):
    text = "|".join([str(x) for x in parts])
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:16]

def get_monthly_rebalance_dates(start_date, end_date):
    days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    out = []
    last_key = None
    for d in days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            out.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return out

def previous_trade_date(date):
    days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(days) < 2:
        return date
    return days[-2].strftime("%Y-%m-%d")

def filter_listing_age(stocks, date, n):
    out = []
    begin_dt = datetime.datetime.strptime(str(date), "%Y-%m-%d")
    cutoff = (begin_dt - datetime.timedelta(days=n)).date()
    for stock in stocks:
        info = get_security_info(stock)
        if info is not None and info.start_date <= cutoff:
            out.append(stock)
    return out

def get_stock_pool(feature_date):
    stock_list = list(get_index_stocks(UNIVERSE_INDEX, feature_date))
    stock_list = unique_keep_order(stock_list)
    if len(stock_list) == 0:
        return []
    st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    if st_data is not None and len(st_data) > 0:
        st_row = st_data.iloc[0]
        valid = []
        for stock in stock_list:
            if stock not in st_row.index:
                valid.append(stock)
            elif pd.isnull(st_row[stock]) or (not bool(st_row[stock])):
                valid.append(stock)
        stock_list = valid
    return filter_listing_age(stock_list, feature_date, MIN_LISTING_DAYS)

def get_factor_data(stock_list, feature_date, factors):
    df = pd.DataFrame(index=stock_list)
    for batch in chunks(factors, FACTOR_BATCH_SIZE):
        factor_data = get_factor_values(securities=stock_list, factors=batch, count=1, end_date=feature_date)
        for f in batch:
            if factor_data is not None and f in factor_data and factor_data[f] is not None and not factor_data[f].empty:
                df[f] = factor_data[f].iloc[0, :].reindex(stock_list)
            else:
                df[f] = np.nan
        gc.collect()
    return downcast_numeric_df(df.replace([np.inf, -np.inf], np.nan))

def get_close_return(stock_list, start_date, end_date):
    rows = []
    for batch in chunks(stock_list, PRICE_BATCH_SIZE):
        price_df = get_price(batch, start_date=start_date, end_date=end_date, frequency="daily", fields=["close"], skip_paused=True, fq="pre", panel=False)
        if price_df is not None and not price_df.empty:
            rows.append(price_df)
    if len(rows) == 0:
        return pd.Series(dtype=float)
    price_df = pd.concat(rows, ignore_index=True)
    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.Series(dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[1] - 1.0

def get_index_return(index_code, start_date, end_date):
    df = get_price(index_code, start_date=start_date, end_date=end_date, frequency="daily", fields=["close"], skip_paused=True, fq="pre")
    if df is None or df.empty or len(df) < 2:
        return np.nan
    return float(df["close"].iloc[-1] / df["close"].iloc[1] - 1.0)

def get_industry_bucket_map(stock_list, feature_date):
    out = {}
    info = get_industry(stock_list, date=feature_date)
    for stock in stock_list:
        bucket = "UNKNOWN"
        one = info.get(stock, {}) if info is not None else {}
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = one.get(key, None)
            if sub:
                bucket = sub.get("industry_code") or sub.get("industry_name") or "UNKNOWN"
                break
        out[stock] = bucket
    return out

def dataset_cache_file(factors):
    key = cache_key([UNIVERSE_INDEX, DATA_START, DATA_END_FOR_LABEL, MIN_LISTING_DAYS, tuple(factors)])
    return os.path.join(CACHE_DIR, "csi800_ml_mainline_dataset_{}.csv".format(key))


In [ ]:
def build_or_load_dataset(factors):
    path = dataset_cache_file(factors)
    if USE_CACHE and (not REBUILD_DATASET) and os.path.exists(path):
        print("load cached dataset ->", path)
        df = pd.read_csv(path)
        for c in ["rebalance_date", "feature_date", "next_date"]:
            df[c] = pd.to_datetime(df[c])
        return downcast_numeric_df(df)

    dates = get_monthly_rebalance_dates(DATA_START, DATA_END_FOR_LABEL)
    print("rebalance dates =", len(dates))
    if os.path.exists(path):
        os.remove(path)
    wrote_header = False
    total_rows = 0

    for i, date in enumerate(dates[:-1]):
        next_date = dates[i + 1]
        feature_date = previous_trade_date(date)
        stock_list = get_stock_pool(feature_date)
        if len(stock_list) == 0:
            continue
        factor_df = get_factor_data(stock_list, feature_date, factors)
        stock_ret = get_close_return(stock_list, date, next_date)
        csi800_ret = get_index_return(BENCHMARK_CSI800, date, next_date)
        alla_ret = get_index_return(BENCHMARK_ALLA, date, next_date)

        data = factor_df.copy()
        data["raw_return_1m"] = stock_ret.reindex(data.index)
        data["benchmark_csi800_1m"] = csi800_ret
        data["benchmark_alla_1m"] = alla_ret
        data["alpha_vs_csi800"] = data["raw_return_1m"] - csi800_ret
        data["alpha_vs_alla"] = data["raw_return_1m"] - alla_ret
        data["stock"] = data.index
        data["rebalance_date"] = date
        data["feature_date"] = feature_date
        data["next_date"] = next_date
        data = data.dropna(subset=["raw_return_1m", "alpha_vs_csi800", "alpha_vs_alla"]).copy()
        if len(data) < 30:
            continue
        industry_map = get_industry_bucket_map(list(data.index), feature_date)
        data["industry_bucket"] = data["stock"].map(industry_map).fillna("UNKNOWN")
        data = downcast_numeric_df(data.reset_index(drop=True))
        data.to_csv(path, mode="a", header=(not wrote_header), index=False)
        wrote_header = True
        total_rows += len(data)
        print("month", date, "rows", len(data), "total", total_rows)
        del data, factor_df
        gc.collect()

    if total_rows == 0:
        raise ValueError("dataset is empty")
    df = pd.read_csv(path)
    for c in ["rebalance_date", "feature_date", "next_date"]:
        df[c] = pd.to_datetime(df[c])
    return downcast_numeric_df(df)


## 4. Score Helpers

In [ ]:
def safe_rank_pct(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index)
    valid = s.dropna()
    if len(valid) <= 1:
        out.loc[valid.index] = 0.5
        return out
    out.loc[valid.index] = valid.rank(pct=True, method="average")
    return out

def rank_by_month(df, col):
    out = pd.Series(np.nan, index=df.index)
    for dt, idx in df.groupby("rebalance_date").groups.items():
        out.loc[idx] = safe_rank_pct(df.loc[idx, col])
    return out

def zscore_by_month(df, col):
    out = pd.Series(0.0, index=df.index)
    for dt, idx in df.groupby("rebalance_date").groups.items():
        s = pd.to_numeric(df.loc[idx, col], errors="coerce")
        std = s.std()
        if std is not None and not pd.isnull(std) and std > 0:
            out.loc[idx] = ((s - s.mean()) / std).clip(-5, 5)
        else:
            out.loc[idx] = 0.0
    return out

def add_manual_scores(df):
    out = df.copy()
    parts = []
    weights = []
    for factor, direction, weight in MANUAL_SCORE_CONFIG:
        if factor not in out.columns:
            continue
        r = rank_by_month(out, factor).fillna(0.5)
        if direction < 0:
            r = 1.0 - r
        parts.append(r)
        weights.append(weight)
    if len(parts) == 0:
        out["manual_rule_score"] = 0.5
    else:
        mat = pd.concat(parts, axis=1).fillna(0.5)
        w = np.asarray(weights, dtype=float)
        w = w / w.sum()
        out["manual_rule_score"] = np.dot(mat.values, w)
    risk_parts = []
    for factor in RISK_PENALTY_FACTORS:
        if factor in out.columns:
            risk_parts.append(rank_by_month(out, factor).fillna(0.5))
    if len(risk_parts):
        out["risk_penalty"] = pd.concat(risk_parts, axis=1).mean(axis=1)
    else:
        out["risk_penalty"] = 0.5
    return out

def safe_corr(a, b):
    s1 = pd.Series(np.asarray(a))
    s2 = pd.Series(np.asarray(b))
    if len(s1) == 0 or len(s2) == 0:
        return np.nan
    if s1.nunique() < 2 or s2.nunique() < 2:
        return np.nan
    return float(s1.corr(s2))

def monthly_rank_ic(df, score_col, target_col):
    vals = []
    idx = []
    for dt, one in df.groupby("rebalance_date"):
        x = safe_rank_pct(one[score_col])
        y = safe_rank_pct(one[target_col])
        vals.append(x.corr(y))
        idx.append(dt)
    return pd.Series(vals, index=idx).dropna()


In [ ]:
def add_hybrid_light_features(df):
    out = df.copy()
    hybrid_cols = []
    for col in HYBRID_LIGHT_RAW_COLS:
        if col in out.columns:
            hybrid_cols.append(col)
    for col in HYBRID_LIGHT_TRANSFORM_COLS:
        if col in out.columns:
            rank_col = "rank_" + col
            z_col = "z_" + col
            out[rank_col] = rank_by_month(out, col).fillna(0.5)
            out[z_col] = zscore_by_month(out, col).fillna(0.0)
            hybrid_cols.append(rank_col)
            hybrid_cols.append(z_col)
    out["board_chinext_flag"] = out["stock"].astype(str).map(lambda x: 1.0 if x.startswith(("300", "301")) else 0.0)
    out["board_star_flag"] = out["stock"].astype(str).map(lambda x: 1.0 if x.startswith("688") else 0.0)
    hybrid_cols.extend(["board_chinext_flag", "board_star_flag"])
    return out, unique_keep_order(hybrid_cols)

def add_rank_industry_features(df, source_cols):
    out = df.copy()
    rank_cols = []
    for col in source_cols:
        if col not in out.columns:
            continue
        rank_col = "ind_rank_" + col
        values = pd.Series(np.nan, index=out.index)
        for dt, month_idx in out.groupby("rebalance_date").groups.items():
            month = out.loc[month_idx]
            for industry, industry_idx in month.groupby("industry_bucket").groups.items():
                values.loc[industry_idx] = safe_rank_pct(out.loc[industry_idx, col])
        out[rank_col] = values.fillna(0.5)
        rank_cols.append(rank_col)
    return out, rank_cols


## 5. LightGBM Helpers

In [ ]:
def calc_single_factor_ic(train_df, cols, target_col):
    rows = []
    for c in cols:
        vals = []
        for dt, d in train_df.groupby("rebalance_date"):
            x = d[c]
            y = d[target_col]
            if x.notnull().sum() < 30 or x.nunique(dropna=True) < 3:
                continue
            vals.append(safe_rank_pct(x).corr(safe_rank_pct(y)))
        vals = pd.Series(vals).dropna()
        avg = float(vals.mean()) if len(vals) else np.nan
        rows.append({"factor": c, "avg_rank_ic": avg, "abs_avg_rank_ic": abs(avg) if not pd.isnull(avg) else np.nan, "months": int(len(vals))})
    return pd.DataFrame(rows)


def select_features_by_corr(train_df, cols, target_col, threshold=0.70):
    cols = [c for c in unique_keep_order(cols) if c in train_df.columns]
    usable = []
    removed = []
    for c in cols:
        s = train_df[c].replace([np.inf, -np.inf], np.nan)
        if s.notnull().sum() < 100 or s.nunique(dropna=True) < 3:
            removed.append(c)
        else:
            usable.append(c)
    if len(usable) == 0:
        return [], removed

    missing_counts = train_df[usable].isnull().sum().to_dict()
    ic_df = calc_single_factor_ic(train_df, usable, target_col).set_index("factor")
    corr_matrix = train_df[usable].corr(method="spearman")

    graph = {}
    for c in usable:
        graph[c] = []
    for i in range(len(usable)):
        for j in range(i + 1, len(usable)):
            c1 = usable[i]
            c2 = usable[j]
            cv = corr_matrix.iloc[i, j]
            if not pd.isnull(cv) and abs(cv) > threshold:
                graph[c1].append(c2)
                graph[c2].append(c1)

    visited = set()
    comps = []
    for c in usable:
        if c in visited:
            continue
        stack = [c]
        comp = []
        visited.add(c)
        while len(stack):
            node = stack.pop()
            comp.append(node)
            for nb in graph.get(node, []):
                if nb not in visited:
                    visited.add(nb)
                    stack.append(nb)
        comps.append(comp)

    keep = []
    corr_removed = []
    for comp in comps:
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            def sort_key(x):
                miss = missing_counts.get(x, 10 ** 9)
                ic = ic_df.loc[x, "abs_avg_rank_ic"] if x in ic_df.index else np.nan
                if pd.isnull(ic):
                    ic = 0
                return (miss, -ic, x)
            comp_sorted = sorted(comp, key=sort_key)
            keep.append(comp_sorted[0])
            corr_removed.extend(comp_sorted[1:])
    return keep, removed + corr_removed


def get_lgb_params():
    params = {
        "objective": "regression",
        "metric": "l2",
        "boosting_type": "gbdt",
        "num_threads": LGB_NUM_THREADS,
        "seed": 42,
        "feature_pre_filter": False,
        "force_col_wise": True,
        "verbose": -1,
    }
    params.update(LGB_PROFILE["params"])
    return params


def train_lgb_fixed(X_train, y_train):
    dtrain = lgb.Dataset(X_train, label=y_train)
    rounds = int(LGB_PROFILE["num_boost_round"])
    model = lgb.train(get_lgb_params(), dtrain, num_boost_round=rounds, verbose_eval=False)
    return model, rounds


def fit_predict_lgb_train_valid(train_df, valid_df, raw_cols, target_col, pred_col):
    feature_cols, removed_cols = select_features_by_corr(train_df, raw_cols, target_col)
    if len(feature_cols) == 0:
        raise ValueError("no features for " + pred_col)
    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    X_valid = valid_df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    fill_values = X_train.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = X_train.fillna(fill_values).fillna(0)
    X_valid = X_valid.fillna(fill_values).fillna(0)
    y_train = train_df[target_col].astype(float).copy()
    model, rounds = train_lgb_fixed(X_train, y_train)
    train_pred = model.predict(X_train, num_iteration=rounds)
    valid_pred = model.predict(X_valid, num_iteration=rounds)
    importance = pd.Series(model.feature_importance(importance_type="gain"), index=feature_cols).sort_values(ascending=False).reset_index()
    importance.columns = ["feature", "gain_importance"]
    importance["pred_col"] = pred_col
    meta = {
        "pred_col": pred_col,
        "target_col": target_col,
        "used_feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "used_features": ",".join(feature_cols),
    }
    return train_pred, valid_pred, importance, meta


def fit_predict_lgb(train_df, valid_df, raw_cols, target_col, pred_col):
    train_pred, valid_pred, importance, meta = fit_predict_lgb_train_valid(train_df, valid_df, raw_cols, target_col, pred_col)
    return valid_pred, importance, meta


## 6. Portfolio Construction

In [ ]:
def board_bucket(stock):
    if stock.startswith("300") or stock.startswith("301"):
        return "ChiNext"
    if stock.startswith("688"):
        return "STAR"
    if stock.startswith("002") or stock.startswith("003"):
        return "SZ_SME"
    if stock.startswith("000") or stock.startswith("001"):
        return "SZ_Main"
    if stock.startswith("600") or stock.startswith("601") or stock.startswith("603") or stock.startswith("605"):
        return "SH_Main"
    return "Other"


def build_industry_targets(one, score_col, candidate_stocks=None, stock_num=None, max_per_industry=None):
    if stock_num is None:
        stock_num = STOCK_NUM
    if max_per_industry is None:
        max_per_industry = MAX_PER_INDUSTRY
    data = one.copy()
    if candidate_stocks is not None:
        data = data[data["stock"].isin(candidate_stocks)].copy()
    cand = data.sort_values(score_col, ascending=False).head(TOP_N_CANDIDATES)
    target = []
    industry_count = {}
    for idx, row in cand.iterrows():
        stock = row["stock"]
        industry = row.get("industry_bucket", "UNKNOWN")
        count = industry_count.get(industry, 0)
        if industry != "UNKNOWN" and count >= max_per_industry:
            continue
        target.append(stock)
        industry_count[industry] = count + 1
        if len(target) >= stock_num:
            break
    if len(target) < stock_num:
        for idx, row in cand.iterrows():
            stock = row["stock"]
            if stock not in target:
                target.append(stock)
            if len(target) >= stock_num:
                break
    return target[:stock_num]


def build_recall_candidates(one, recall_cols, recall_n):
    target = []
    for col in recall_cols:
        if col not in one.columns:
            continue
        names = one.sort_values(col, ascending=False).head(recall_n)["stock"].tolist()
        for s in names:
            if s not in target:
                target.append(s)
    return target


def build_group_recall_candidates(one, group_score_cols, base_score_col, recall_n):
    return build_recall_candidates(one, [base_score_col] + group_score_cols, recall_n)


def build_recall_mask(df, recall_cols, recall_n):
    mask = pd.Series(False, index=df.index)
    for dt, one in df.groupby("rebalance_date"):
        candidates = build_recall_candidates(one, recall_cols, recall_n)
        mask.loc[one.index] = one["stock"].isin(candidates).values
    return mask


def select_targets(one, strategy_name, score_col, portfolio_profile, group_score_cols):
    candidate_stocks = None
    if strategy_name == "factor_group_recall_lgb":
        candidate_stocks = build_group_recall_candidates(one, group_score_cols, "baseline_lgb_score", GROUP_RECALL_N)
    if strategy_name.startswith("recall_rerank_lgb"):
        candidate_stocks = build_recall_candidates(one, RERANK_RECALL_COLS, RERANK_RECALL_N)
    data = one.copy()
    if candidate_stocks is not None:
        data = data[data["stock"].isin(candidate_stocks)].copy()
    if portfolio_profile == "top10":
        return data.sort_values(score_col, ascending=False).head(10)["stock"].tolist()
    if portfolio_profile == "top20":
        return data.sort_values(score_col, ascending=False).head(20)["stock"].tolist()
    if portfolio_profile == "industry_top10":
        return build_industry_targets(one, score_col, candidate_stocks=candidate_stocks)
    raise ValueError("unknown portfolio profile " + portfolio_profile)


def monthly_portfolio_returns(score_df, strategy_name, score_col, portfolio_profile, group_score_cols):
    rows = []
    for dt, one in score_df.groupby("rebalance_date"):
        target = select_targets(one, strategy_name, score_col, portfolio_profile, group_score_cols)
        top = one[one["stock"].isin(target)].copy()
        if top.empty:
            continue
        boards = pd.Series([board_bucket(s) for s in target]).value_counts().to_dict()
        row = {
            "rebalance_date": dt,
            "strategy_name": strategy_name,
            "score_col": score_col,
            "portfolio_profile": portfolio_profile,
            "target_count": len(top),
            "raw_return_1m": top["raw_return_1m"].mean(),
            "benchmark_csi800_1m": top["benchmark_csi800_1m"].iloc[0],
            "benchmark_alla_1m": top["benchmark_alla_1m"].iloc[0],
            "alpha_vs_csi800": top["alpha_vs_csi800"].mean(),
            "alpha_vs_alla": top["alpha_vs_alla"].mean(),
            "targets": ",".join(target),
            "board_chinext": boards.get("ChiNext", 0),
            "board_star": boards.get("STAR", 0),
            "board_main": boards.get("SH_Main", 0) + boards.get("SZ_Main", 0) + boards.get("SZ_SME", 0),
        }
        rows.append(row)
    out = pd.DataFrame(rows).sort_values("rebalance_date")
    if not out.empty:
        out["cum_ret"] = (1.0 + out["raw_return_1m"]).cumprod() - 1.0
        out["cum_csi800"] = (1.0 + out["benchmark_csi800_1m"]).cumprod() - 1.0
        out["cum_excess_csi800"] = (1.0 + out["alpha_vs_csi800"]).cumprod() - 1.0
    return out


def combine_sleeve_returns(ret_lookup, sleeve_name, sleeve_parts):
    frames = []
    for strategy_name, profile, weight in sleeve_parts:
        one = ret_lookup.get((strategy_name, profile), pd.DataFrame()).copy()
        if one.empty:
            continue
        keep_cols = [
            "rebalance_date", "raw_return_1m", "benchmark_csi800_1m", "benchmark_alla_1m",
            "alpha_vs_csi800", "alpha_vs_alla", "board_chinext", "board_star", "board_main", "targets",
        ]
        one = one[keep_cols].copy()
        one["weight"] = float(weight)
        one["component"] = strategy_name + ":" + profile
        frames.append(one)
    if len(frames) == 0:
        return pd.DataFrame()
    data = pd.concat(frames, ignore_index=True)
    rows = []
    for dt, gdf in data.groupby("rebalance_date"):
        wsum = gdf["weight"].sum()
        if wsum <= 0:
            continue
        row = {
            "rebalance_date": dt,
            "strategy_name": sleeve_name,
            "score_col": "sleeve_weighted_components",
            "portfolio_profile": "sleeve",
            "target_count": len(set(",".join(gdf["targets"].astype(str).tolist()).split(","))),
            "raw_return_1m": float((gdf["raw_return_1m"] * gdf["weight"]).sum() / wsum),
            "benchmark_csi800_1m": float(gdf["benchmark_csi800_1m"].iloc[0]),
            "benchmark_alla_1m": float(gdf["benchmark_alla_1m"].iloc[0]),
            "alpha_vs_csi800": float((gdf["alpha_vs_csi800"] * gdf["weight"]).sum() / wsum),
            "alpha_vs_alla": float((gdf["alpha_vs_alla"] * gdf["weight"]).sum() / wsum),
            "board_chinext": float((gdf["board_chinext"] * gdf["weight"]).sum() / wsum),
            "board_star": float((gdf["board_star"] * gdf["weight"]).sum() / wsum),
            "board_main": float((gdf["board_main"] * gdf["weight"]).sum() / wsum),
            "targets": " | ".join((gdf["component"] + "=" + gdf["targets"].astype(str)).tolist()),
        }
        rows.append(row)
    out = pd.DataFrame(rows).sort_values("rebalance_date")
    if not out.empty:
        out["cum_ret"] = (1.0 + out["raw_return_1m"]).cumprod() - 1.0
        out["cum_csi800"] = (1.0 + out["benchmark_csi800_1m"]).cumprod() - 1.0
        out["cum_excess_csi800"] = (1.0 + out["alpha_vs_csi800"]).cumprod() - 1.0
    return out


def calc_max_drawdown(ret_series):
    equity = (1.0 + ret_series).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())


def summarize_monthly(ret_df):
    if ret_df is None or ret_df.empty:
        return {}
    return {
        "months": int(len(ret_df)),
        "cum_ret": float((1.0 + ret_df["raw_return_1m"]).prod() - 1.0),
        "cum_csi800": float((1.0 + ret_df["benchmark_csi800_1m"]).prod() - 1.0),
        "cum_excess_csi800": float((1.0 + ret_df["alpha_vs_csi800"]).prod() - 1.0),
        "mean_monthly_excess": float(ret_df["alpha_vs_csi800"].mean()),
        "win_rate": float((ret_df["alpha_vs_csi800"] > 0).mean()),
        "max_drawdown": calc_max_drawdown(ret_df["raw_return_1m"]),
        "avg_chinext": float(ret_df["board_chinext"].mean()),
        "avg_star": float(ret_df["board_star"].mean()),
        "avg_main": float(ret_df["board_main"].mean()),
    }


def calc_turnover(ret_df):
    if ret_df is None or ret_df.empty:
        return np.nan
    prev = None
    turns = []
    for idx, row in ret_df.sort_values("rebalance_date").iterrows():
        cur = [x for x in str(row["targets"]).replace(" | ", ",").split(",") if x]
        if prev is not None and len(cur) > 0:
            overlap = len(set(cur) & set(prev))
            turns.append(1.0 - overlap / float(len(cur)))
        prev = cur
    if len(turns) == 0:
        return np.nan
    return float(np.mean(turns))


def stress_without_top_months(ret_df, n):
    if ret_df is None or ret_df.empty:
        return np.nan
    ex = ret_df["alpha_vs_csi800"]
    drop_idx = ex.sort_values(ascending=False).head(n).index
    rem = ret_df.drop(drop_idx)
    if rem.empty:
        return np.nan
    return float((1.0 + rem["alpha_vs_csi800"]).prod() - 1.0)


def average_overlap(ret_a, ret_b):
    if ret_a is None or ret_b is None or ret_a.empty or ret_b.empty:
        return np.nan
    a = ret_a.set_index("rebalance_date")
    b = ret_b.set_index("rebalance_date")
    vals = []
    for dt in sorted(set(a.index) & set(b.index)):
        ta = [x for x in str(a.loc[dt, "targets"]).replace(" | ", ",").split(",") if x]
        tb = [x for x in str(b.loc[dt, "targets"]).replace(" | ", ",").split(",") if x]
        if len(ta) > 0:
            vals.append(len(set(ta) & set(tb)) / float(len(ta)))
    return float(np.mean(vals)) if len(vals) else np.nan


## 7. Build Dataset

In [ ]:
df_raw = build_or_load_dataset(ALL_FACTORS)
df_scored = add_manual_scores(df_raw)
df_hybrid, HYBRID_LIGHT_FEATURE_COLS = add_hybrid_light_features(df_scored)
df_all, RANK_INDUSTRY_FEATURE_COLS = add_rank_industry_features(df_hybrid, FACTOR_SET_V22A_37)

train_df = df_all[(df_all["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df_all["rebalance_date"] <= pd.Timestamp(TRAIN_END)) & (df_all["next_date"] <= pd.Timestamp(TRAIN_END))].copy()
valid_df = df_all[(df_all["rebalance_date"] >= pd.Timestamp(VALID_START)) & (df_all["rebalance_date"] <= pd.Timestamp(VALID_END))].copy()

print("df_all shape:", df_all.shape)
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]).to_string())
print("train rows/months:", len(train_df), train_df["rebalance_date"].nunique())
print("valid rows/months:", len(valid_df), valid_df["rebalance_date"].nunique())
print("hybrid_light_features:", len(HYBRID_LIGHT_FEATURE_COLS))
print("rank_industry_features:", len(RANK_INDUSTRY_FEATURE_COLS))


## 8. Train Family Scores

In [ ]:
base_output_cols = [
    "stock", "rebalance_date", "next_date", "industry_bucket",
    "raw_return_1m", "benchmark_csi800_1m", "benchmark_alla_1m",
    "alpha_vs_csi800", "alpha_vs_alla", "manual_rule_score", "risk_penalty",
]
train_score_df = train_df[base_output_cols].copy()
score_df = valid_df[base_output_cols].copy()

importance_parts = []
model_meta = []

base_train_pred, base_valid_pred, baseline_imp, baseline_meta = fit_predict_lgb_train_valid(train_df, valid_df, FACTOR_SET_V22A_37, "alpha_vs_alla", "baseline_lgb_score")
train_score_df["baseline_lgb_score"] = base_train_pred
score_df["baseline_lgb_score"] = base_valid_pred
baseline_imp["model_family"] = "raw_v22a_lgb_alpha_vs_alla"
baseline_meta["model_family"] = "raw_v22a_lgb_alpha_vs_alla"
importance_parts.append(baseline_imp)
model_meta.append(baseline_meta)

hybrid_train_pred, hybrid_valid_pred, hybrid_imp, hybrid_meta = fit_predict_lgb_train_valid(train_df, valid_df, HYBRID_LIGHT_FEATURE_COLS, "alpha_vs_csi800", "hybrid_light_lgb_score")
train_score_df["hybrid_light_lgb_score"] = hybrid_train_pred
score_df["hybrid_light_lgb_score"] = hybrid_valid_pred
hybrid_imp["model_family"] = "hybrid_light_lgb_alpha_vs_csi800"
hybrid_meta["model_family"] = "hybrid_light_lgb_alpha_vs_csi800"
importance_parts.append(hybrid_imp)
model_meta.append(hybrid_meta)

rank_csi_train_pred, rank_csi_valid_pred, rank_csi_imp, rank_csi_meta = fit_predict_lgb_train_valid(train_df, valid_df, RANK_INDUSTRY_FEATURE_COLS, "alpha_vs_csi800", "rank_industry_csi800_score")
train_score_df["rank_industry_csi800_score"] = rank_csi_train_pred
score_df["rank_industry_csi800_score"] = rank_csi_valid_pred
rank_csi_imp["model_family"] = "rank_industry_lgb_alpha_vs_csi800"
rank_csi_meta["model_family"] = "rank_industry_lgb_alpha_vs_csi800"
importance_parts.append(rank_csi_imp)
model_meta.append(rank_csi_meta)

rank_alla_train_pred, rank_alla_valid_pred, rank_alla_imp, rank_alla_meta = fit_predict_lgb_train_valid(train_df, valid_df, RANK_INDUSTRY_FEATURE_COLS, "alpha_vs_alla", "rank_industry_alla_score")
train_score_df["rank_industry_alla_score"] = rank_alla_train_pred
score_df["rank_industry_alla_score"] = rank_alla_valid_pred
rank_alla_imp["model_family"] = "rank_industry_lgb_alpha_vs_alla"
rank_alla_meta["model_family"] = "rank_industry_lgb_alpha_vs_alla"
importance_parts.append(rank_alla_imp)
model_meta.append(rank_alla_meta)

group_score_cols = []
for group_name in ["value_cashflow", "quality_profit", "growth_balance", "momentum_risk", "technical_volume"]:
    cols = FACTOR_GROUPS[group_name]
    pred_col = "group_score_" + group_name
    train_pred, valid_pred, imp, meta = fit_predict_lgb_train_valid(train_df, valid_df, cols, "alpha_vs_alla", pred_col)
    train_score_df[pred_col] = train_pred
    score_df[pred_col] = valid_pred
    group_score_cols.append(pred_col)
    imp["model_family"] = "factor_group_recall_lgb"
    meta["model_family"] = "factor_group_recall_lgb"
    importance_parts.append(imp)
    model_meta.append(meta)

for frame in [train_score_df, score_df]:
    frame["baseline_rank"] = rank_by_month(frame, "baseline_lgb_score").fillna(0.5)
    frame["hybrid_rank"] = rank_by_month(frame, "hybrid_light_lgb_score").fillna(0.5)
    frame["rank_industry_csi800_rank"] = rank_by_month(frame, "rank_industry_csi800_score").fillna(0.5)
    frame["rank_industry_alla_rank"] = rank_by_month(frame, "rank_industry_alla_score").fillna(0.5)
    frame["manual_rank"] = rank_by_month(frame, "manual_rule_score").fillna(0.5)
    frame["risk_rank"] = rank_by_month(frame, "risk_penalty").fillna(0.5)
    group_rank_cols = []
    for col in group_score_cols:
        rcol = col + "_rank"
        frame[rcol] = rank_by_month(frame, col).fillna(0.5)
        group_rank_cols.append(rcol)
    frame["group_mean_rank"] = frame[group_rank_cols].mean(axis=1)
    frame["factor_group_recall_lgb_score"] = 0.65 * frame["baseline_rank"] + 0.35 * frame["group_mean_rank"]
    frame["v3_lgb_blend_score"] = 0.55 * frame["baseline_rank"] + 0.30 * frame["manual_rank"] + 0.15 * frame["hybrid_rank"]

RERANK_RECALL_COLS = [
    "baseline_lgb_score", "rank_industry_csi800_score", "rank_industry_alla_score",
    "factor_group_recall_lgb_score", "v3_lgb_blend_score", "manual_rule_score",
]
RERANK_FEATURE_COLS = unique_keep_order(
    RANK_INDUSTRY_FEATURE_COLS +
    ["baseline_rank", "hybrid_rank", "manual_rank", "risk_rank", "group_mean_rank", "rank_industry_csi800_rank", "rank_industry_alla_rank"] +
    group_score_cols
)

train_rerank_df = train_df.copy()
valid_rerank_df = valid_df.copy()
score_cols_to_merge = [c for c in train_score_df.columns if c not in base_output_cols]
for c in score_cols_to_merge:
    train_rerank_df[c] = train_score_df[c].values
    valid_rerank_df[c] = score_df[c].values
train_recall_mask = build_recall_mask(train_rerank_df, RERANK_RECALL_COLS, RERANK_RECALL_N)
train_rerank_sample = train_rerank_df[train_recall_mask].copy()
print("rerank train sample:", len(train_rerank_sample), "from", len(train_rerank_df))

rerank_alla_train_pred, rerank_alla_valid_pred, rerank_alla_imp, rerank_alla_meta = fit_predict_lgb_train_valid(train_rerank_sample, valid_rerank_df, RERANK_FEATURE_COLS, "alpha_vs_alla", "recall_rerank_alla_score")
score_df["recall_rerank_alla_score"] = rerank_alla_valid_pred
rerank_alla_imp["model_family"] = "recall_rerank_lgb_alpha_vs_alla"
rerank_alla_meta["model_family"] = "recall_rerank_lgb_alpha_vs_alla"
importance_parts.append(rerank_alla_imp)
model_meta.append(rerank_alla_meta)

rerank_csi_train_pred, rerank_csi_valid_pred, rerank_csi_imp, rerank_csi_meta = fit_predict_lgb_train_valid(train_rerank_sample, valid_rerank_df, RERANK_FEATURE_COLS, "alpha_vs_csi800", "recall_rerank_csi800_score")
score_df["recall_rerank_csi800_score"] = rerank_csi_valid_pred
rerank_csi_imp["model_family"] = "recall_rerank_lgb_alpha_vs_csi800"
rerank_csi_meta["model_family"] = "recall_rerank_lgb_alpha_vs_csi800"
importance_parts.append(rerank_csi_imp)
model_meta.append(rerank_csi_meta)

importance_df = pd.concat(importance_parts, ignore_index=True)
model_meta_df = pd.DataFrame(model_meta)

print("model meta:")
print(model_meta_df[["model_family", "pred_col", "target_col", "used_feature_count", "removed_feature_count"]].to_string(index=False))
print("score_df shape:", score_df.shape)


## 9. Evaluate Strategy Families

In [ ]:
STRATEGIES = [
    ("manual_v3", "manual_rule_score", "manual_v3"),
    ("raw_v22a_lgb_alpha_vs_alla", "baseline_lgb_score", "baseline"),
    ("hybrid_light_lgb_alpha_vs_csi800", "hybrid_light_lgb_score", "hybrid_light"),
    ("rank_industry_lgb_alpha_vs_csi800", "rank_industry_csi800_score", "rank_industry"),
    ("rank_industry_lgb_alpha_vs_alla", "rank_industry_alla_score", "rank_industry"),
    ("factor_group_recall_lgb", "factor_group_recall_lgb_score", "factor_group_recall"),
    ("recall_rerank_lgb_alpha_vs_alla", "recall_rerank_alla_score", "recall_rerank"),
    ("recall_rerank_lgb_alpha_vs_csi800", "recall_rerank_csi800_score", "recall_rerank"),
    ("v3_lgb_blend", "v3_lgb_blend_score", "blend"),
]
PORTFOLIO_PROFILES = ["top10", "top20", "industry_top10"]
SLEEVE_DEFS = [
    ("sleeve_60_base_top10_40_v3blend_top20", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("v3_lgb_blend", "top20", 0.40)]),
    ("sleeve_60_base_top10_40_group_industry_top10", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("factor_group_recall_lgb", "industry_top10", 0.40)]),
    ("sleeve_60_base_top10_40_rankind_csi_top20", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("rank_industry_lgb_alpha_vs_csi800", "top20", 0.40)]),
    ("sleeve_60_base_top10_40_rerank_alla_top10", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("recall_rerank_lgb_alpha_vs_alla", "top10", 0.40)]),
    ("sleeve_50_base_top20_50_v3blend_top20", [("raw_v22a_lgb_alpha_vs_alla", "top20", 0.50), ("v3_lgb_blend", "top20", 0.50)]),
]

summary_rows = []
monthly_parts = []
rank_ic_rows = []
ret_lookup = {}
for strategy_name, score_col, family_tag in STRATEGIES:
    ic_csi = monthly_rank_ic(score_df, score_col, "alpha_vs_csi800")
    ic_alla = monthly_rank_ic(score_df, score_col, "alpha_vs_alla")
    for portfolio_profile in PORTFOLIO_PROFILES:
        ret_df = monthly_portfolio_returns(score_df, strategy_name, score_col, portfolio_profile, group_score_cols)
        ret_lookup[(strategy_name, portfolio_profile)] = ret_df
        row = {
            "strategy_name": strategy_name,
            "family_tag": family_tag,
            "score_col": score_col,
            "portfolio_profile": portfolio_profile,
            "rank_ic_csi800_mean": float(ic_csi.mean()) if len(ic_csi) else np.nan,
            "rank_ic_csi800_ir": float(ic_csi.mean() / ic_csi.std() * np.sqrt(12)) if len(ic_csi) and ic_csi.std() > 0 else np.nan,
            "rank_ic_alla_mean": float(ic_alla.mean()) if len(ic_alla) else np.nan,
            "rank_ic_alla_ir": float(ic_alla.mean() / ic_alla.std() * np.sqrt(12)) if len(ic_alla) and ic_alla.std() > 0 else np.nan,
            "avg_turnover": calc_turnover(ret_df),
            "drop_top1_excess": stress_without_top_months(ret_df, 1),
            "drop_top3_excess": stress_without_top_months(ret_df, 3),
        }
        row.update(summarize_monthly(ret_df))
        summary_rows.append(row)
        if not ret_df.empty:
            monthly_parts.append(ret_df)
    rank_ic_rows.append(pd.DataFrame({
        "rebalance_date": ic_csi.index,
        "strategy_name": strategy_name,
        "family_tag": family_tag,
        "score_col": score_col,
        "rank_ic_csi800": ic_csi.values,
        "rank_ic_alla": ic_alla.reindex(ic_csi.index).values,
    }))

for sleeve_name, sleeve_parts in SLEEVE_DEFS:
    ret_df = combine_sleeve_returns(ret_lookup, sleeve_name, sleeve_parts)
    ret_lookup[(sleeve_name, "sleeve")] = ret_df
    row = {
        "strategy_name": sleeve_name,
        "family_tag": "sleeve_portfolio",
        "score_col": "sleeve_weighted_components",
        "portfolio_profile": "sleeve",
        "rank_ic_csi800_mean": np.nan,
        "rank_ic_csi800_ir": np.nan,
        "rank_ic_alla_mean": np.nan,
        "rank_ic_alla_ir": np.nan,
        "avg_turnover": calc_turnover(ret_df),
        "drop_top1_excess": stress_without_top_months(ret_df, 1),
        "drop_top3_excess": stress_without_top_months(ret_df, 3),
    }
    row.update(summarize_monthly(ret_df))
    summary_rows.append(row)
    if not ret_df.empty:
        monthly_parts.append(ret_df)

summary_df = pd.DataFrame(summary_rows).sort_values(["cum_excess_csi800", "rank_ic_csi800_mean"], ascending=[False, False])
monthly_df = pd.concat(monthly_parts, ignore_index=True) if len(monthly_parts) else pd.DataFrame()
rank_ic_df = pd.concat(rank_ic_rows, ignore_index=True) if len(rank_ic_rows) else pd.DataFrame()

baseline_top10 = ret_lookup.get(("raw_v22a_lgb_alpha_vs_alla", "top10"), pd.DataFrame())
summary_df["overlap_with_baseline_top10"] = np.nan
for idx, row in summary_df.iterrows():
    key = (row["strategy_name"], row["portfolio_profile"])
    summary_df.loc[idx, "overlap_with_baseline_top10"] = average_overlap(ret_lookup.get(key, pd.DataFrame()), baseline_top10)

summary_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_summary.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_monthly.csv"), index=False)
rank_ic_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_rank_ic.csv"), index=False)
importance_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_importance.csv"), index=False)
model_meta_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_model_meta.csv"), index=False)
score_df.to_csv(os.path.join(OUT_DIR, "mainline_family_v3_scores.csv"), index=False)

print("saved outputs ->", OUT_DIR)
display(summary_df)


## 10. Final Summary

In [ ]:
print("================ CSI800 ML MAINLINE FAMILY V3 FULL FAMILIES SUMMARY ================")
print("Protocol: train <= 2024-12-31, evaluate 2025+ OOS. 2025+ is not used for family weight tuning.")
print("Baseline anchor: raw v22a_37 + alpha_vs_alla + fixed_regularized_120")
print("New blocks: rank_industry_lgb, true recall_rerank_lgb, fixed sleeve portfolios")
print("data shape:", df_all.shape)
print("date range:")
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]).to_string())
print("train rows/months:", len(train_df), train_df["rebalance_date"].nunique())
print("valid rows/months:", len(valid_df), valid_df["rebalance_date"].nunique())
print("rank_industry_features:", len(RANK_INDUSTRY_FEATURE_COLS))
print("rerank_recall_cols:", RERANK_RECALL_COLS)
print("rerank_feature_count:", len(RERANK_FEATURE_COLS))

cols = [
    "strategy_name", "family_tag", "portfolio_profile", "months",
    "cum_ret", "cum_csi800", "cum_excess_csi800", "mean_monthly_excess", "win_rate", "max_drawdown",
    "rank_ic_csi800_mean", "rank_ic_csi800_ir", "rank_ic_alla_mean", "rank_ic_alla_ir",
    "avg_turnover", "avg_chinext", "avg_star", "avg_main", "drop_top1_excess", "drop_top3_excess",
    "overlap_with_baseline_top10",
]
print("")
print("Strategy family comparison:")
print(summary_df[cols].to_string(index=False))

print("")
print("Family pivot by cumulative CSI800 excess:")
print(summary_df.pivot_table(index="strategy_name", columns="portfolio_profile", values="cum_excess_csi800", aggfunc="max").round(4).to_string())

print("")
print("Best monthly tail:")
best = summary_df.iloc[0]
bm = monthly_df[(monthly_df["strategy_name"] == best["strategy_name"]) & (monthly_df["portfolio_profile"] == best["portfolio_profile"])].sort_values("rebalance_date")
print(bm.tail(8).to_string(index=False))

print("")
print("Model meta:")
print(model_meta_df[["model_family", "pred_col", "target_col", "used_feature_count", "removed_feature_count", "used_features"]].to_string(index=False))

print("")
print("Top importance by model:")
for pred_col in model_meta_df["pred_col"].tolist():
    print("---", pred_col, "---")
    print(importance_df[importance_df["pred_col"] == pred_col].head(15).to_string(index=False))

print("")
print("Decision rules:")
print("1. Mainline replacement must beat or closely match baseline top10 and improve max drawdown or drop-top stress.")
print("2. Rank-industry or recall-rerank wins only if top10/top20/industry_top10 point in the same direction.")
print("3. Sleeve wins only if it keeps at least 70% of baseline excess while materially improving drawdown/stress/exposure.")
print("4. No 2025+ weight optimization. If fixed sleeve fails, discard instead of tuning.")


## 11. Export Walk-Forward PKL Model Bundles

导出 JoinQuant 回测文件可直接加载的 PKL 模型包。这里不导出离线持仓表，回测端会按 bundle 内的 scorer 和 sleeve_components 在线算分、选股、下单。


In [ ]:
import pickle

V3_EXPORT_DIR = OUT_DIR
V3_EXPORT_WINDOW = "expanding_walk_forward"

V3_EXPORT_PRED_SPECS = [
    ("baseline_lgb_score", FACTOR_SET_V22A_37, "alpha_vs_alla"),
    ("hybrid_light_lgb_score", HYBRID_LIGHT_FEATURE_COLS, "alpha_vs_csi800"),
    ("group_score_value_cashflow", FACTOR_GROUPS["value_cashflow"], "alpha_vs_alla"),
    ("group_score_quality_profit", FACTOR_GROUPS["quality_profit"], "alpha_vs_alla"),
    ("group_score_growth_balance", FACTOR_GROUPS["growth_balance"], "alpha_vs_alla"),
    ("group_score_momentum_risk", FACTOR_GROUPS["momentum_risk"], "alpha_vs_alla"),
    ("group_score_technical_volume", FACTOR_GROUPS["technical_volume"], "alpha_vs_alla"),
]

V3_GROUP_SCORE_COLS = [x[0] for x in V3_EXPORT_PRED_SPECS if x[0].startswith("group_score_")]

V3_EXPORT_VARIANTS = [
    {
        "candidate_name": "v3_baseline_top10",
        "file_name": "model_csi800_lgb_v3_baseline_top10_walkforward.pkl",
        "strategy_name": "raw_v22a_lgb_alpha_vs_alla",
        "portfolio_profile": "top10",
        "score_col": "baseline_lgb_score",
        "required_pred_cols": ["baseline_lgb_score"],
        "stock_num": 10,
        "sleeve_components": [],
    },
    {
        "candidate_name": "v3_baseline_top20",
        "file_name": "model_csi800_lgb_v3_baseline_top20_walkforward.pkl",
        "strategy_name": "raw_v22a_lgb_alpha_vs_alla",
        "portfolio_profile": "top20",
        "score_col": "baseline_lgb_score",
        "required_pred_cols": ["baseline_lgb_score"],
        "stock_num": 20,
        "sleeve_components": [],
    },
    {
        "candidate_name": "v3_blend_top20",
        "file_name": "model_csi800_lgb_v3_v3blend_top20_walkforward.pkl",
        "strategy_name": "v3_lgb_blend",
        "portfolio_profile": "top20",
        "score_col": "v3_lgb_blend_score",
        "required_pred_cols": ["baseline_lgb_score", "hybrid_light_lgb_score"],
        "stock_num": 20,
        "sleeve_components": [],
    },
    {
        "candidate_name": "v3_sleeve_60_base10_40_v3blend20",
        "file_name": "model_csi800_lgb_v3_sleeve_60_base10_40_v3blend20_walkforward.pkl",
        "strategy_name": "sleeve_60_base_top10_40_v3blend_top20",
        "portfolio_profile": "sleeve",
        "score_col": "sleeve_weighted_components",
        "required_pred_cols": ["baseline_lgb_score", "hybrid_light_lgb_score"],
        "stock_num": 30,
        "sleeve_components": [
            {"name": "base_top10", "score_col": "baseline_lgb_score", "portfolio_profile": "top10", "stock_num": 10, "weight": 0.60},
            {"name": "v3blend_top20", "score_col": "v3_lgb_blend_score", "portfolio_profile": "top20", "stock_num": 20, "weight": 0.40},
        ],
    },
    {
        "candidate_name": "v3_sleeve_50_base20_50_v3blend20",
        "file_name": "model_csi800_lgb_v3_sleeve_50_base20_50_v3blend20_walkforward.pkl",
        "strategy_name": "sleeve_50_base_top20_50_v3blend_top20",
        "portfolio_profile": "sleeve",
        "score_col": "sleeve_weighted_components",
        "required_pred_cols": ["baseline_lgb_score", "hybrid_light_lgb_score"],
        "stock_num": 40,
        "sleeve_components": [
            {"name": "base_top20", "score_col": "baseline_lgb_score", "portfolio_profile": "top20", "stock_num": 20, "weight": 0.50},
            {"name": "v3blend_top20", "score_col": "v3_lgb_blend_score", "portfolio_profile": "top20", "stock_num": 20, "weight": 0.50},
        ],
    },
    {
        "candidate_name": "v3_sleeve_60_base10_40_groupind10",
        "file_name": "model_csi800_lgb_v3_sleeve_60_base10_40_groupind10_walkforward.pkl",
        "strategy_name": "sleeve_60_base_top10_40_group_industry_top10",
        "portfolio_profile": "sleeve",
        "score_col": "sleeve_weighted_components",
        "required_pred_cols": ["baseline_lgb_score"] + V3_GROUP_SCORE_COLS,
        "stock_num": 20,
        "sleeve_components": [
            {"name": "base_top10", "score_col": "baseline_lgb_score", "portfolio_profile": "top10", "stock_num": 10, "weight": 0.60},
            {
                "name": "group_industry_top10", "score_col": "factor_group_recall_lgb_score",
                "portfolio_profile": "industry_top10", "stock_num": 10, "weight": 0.40,
                "group_recall_n": GROUP_RECALL_N, "industry_cap": MAX_PER_INDUSTRY,
            },
        ],
    },
]


def raw_source_for_model_feature(feature):
    if feature.startswith("rank_"):
        return feature[5:]
    if feature.startswith("z_"):
        return feature[2:]
    if feature in ["board_chinext_flag", "board_star_flag"]:
        return None
    return feature


def raw_sources_for_model_features(features):
    out = []
    for feature in features:
        src = raw_source_for_model_feature(feature)
        if src is not None:
            out.append(src)
    return unique_keep_order(out)


def train_export_model_spec(train_df, raw_cols, target_col, pred_col):
    feature_cols, removed_cols = select_features_by_corr(train_df, raw_cols, target_col)
    if len(feature_cols) == 0:
        raise ValueError("no features for " + pred_col)
    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    fill_values = X_train.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = X_train.fillna(fill_values).fillna(0)
    y_train = train_df[target_col].astype(float).copy()
    model, rounds = train_lgb_fixed(X_train, y_train)
    return {
        "model": model,
        "feature_cols": list(feature_cols),
        "fill_values": dict((k, float(v)) for k, v in fill_values.to_dict().items()),
        "removed_feature_cols": list(removed_cols),
        "target_col": target_col,
        "num_boost_round": int(rounds),
    }


export_valid_months = sorted(df_all[
    (df_all["rebalance_date"] >= pd.Timestamp(VALID_START)) &
    (df_all["rebalance_date"] <= pd.Timestamp(VALID_END))
]["rebalance_date"].unique())

v3_model_bank = {}
v3_meta_rows = []
all_model_features = []
all_required_factors = []

for m in export_valid_months:
    m = pd.Timestamp(m)
    date_tag = m.strftime("%Y%m%d")
    month_train_df = df_all[(df_all["rebalance_date"] < m) & (df_all["next_date"] <= m)].copy()
    if len(month_train_df) < 10000:
        print("skip export month", date_tag, "train rows", len(month_train_df))
        continue
    print("export v3 model bank month", date_tag, "train", len(month_train_df))
    v3_model_bank[date_tag] = {}

    for pred_col, raw_cols, target_col in V3_EXPORT_PRED_SPECS:
        spec = train_export_model_spec(month_train_df, raw_cols, target_col, pred_col)
        v3_model_bank[date_tag][pred_col] = spec
        all_model_features.extend(spec["feature_cols"])
        all_required_factors.extend(raw_sources_for_model_features(spec["feature_cols"]))
        v3_meta_rows.append({
            "model_date_tag": date_tag,
            "pred_col": pred_col,
            "target_col": target_col,
            "feature_count": len(spec["feature_cols"]),
            "removed_count": len(spec["removed_feature_cols"]),
            "features": ",".join(spec["feature_cols"]),
            "removed_features": ",".join(spec["removed_feature_cols"]),
        })
        gc.collect()

for item in MANUAL_SCORE_CONFIG:
    all_required_factors.append(item[0])
all_required_factors.extend(RISK_PENALTY_FACTORS)
all_required_factors = unique_keep_order(all_required_factors)
all_model_features = unique_keep_order(all_model_features)

export_rows = []
for variant in V3_EXPORT_VARIANTS:
    required_pred_cols = list(variant["required_pred_cols"])
    variant_models = {}
    for date_tag in sorted(v3_model_bank.keys()):
        variant_models[date_tag] = {}
        for pred_col in required_pred_cols:
            variant_models[date_tag][pred_col] = v3_model_bank[date_tag][pred_col]

    variant_group_cols = [x for x in required_pred_cols if x.startswith("group_score_")]
    bundle = {
        "objective": "csi800_lgb_named_strategy_pkl",
        "research_version": "mainline_family_v3_walkforward_pkl",
        "window_name": V3_EXPORT_WINDOW,
        "candidate_name": variant["candidate_name"],
        "strategy_name": variant["strategy_name"],
        "portfolio_profile": variant["portfolio_profile"],
        "score_col": variant["score_col"],
        "sleeve_components": variant["sleeve_components"],
        "models": variant_models,
        "model_date_tags": sorted(variant_models.keys()),
        "pred_cols": required_pred_cols,
        "group_score_cols": variant_group_cols,
        "manual_score_config": MANUAL_SCORE_CONFIG,
        "risk_penalty_factors": RISK_PENALTY_FACTORS,
        "all_required_factors": all_required_factors,
        "all_model_features": all_model_features,
        "universe_index": UNIVERSE_INDEX,
        "benchmark": BENCHMARK_CSI800,
        "stock_num": int(variant["stock_num"]),
        "industry_cap": None,
        "max_growth_board": None,
        "keep_rank_n": None,
        "group_recall_n": GROUP_RECALL_N if len(variant_group_cols) else None,
        "max_per_industry": MAX_PER_INDUSTRY,
        "score_formula": "v3: baseline_rank/manual_rank/hybrid_rank; group: 0.65*baseline_rank+0.35*group_mean_rank; sleeve: component weighted buckets",
        "train_start": TRAIN_START,
        "valid_start": VALID_START,
        "valid_end": VALID_END,
        "lgb_profile": LGB_PROFILE,
    }

    path = os.path.join(V3_EXPORT_DIR, variant["file_name"])
    with open(path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)
    export_rows.append({
        "candidate_name": variant["candidate_name"],
        "file_name": variant["file_name"],
        "path": path,
        "strategy_name": variant["strategy_name"],
        "portfolio_profile": variant["portfolio_profile"],
        "pred_col_count": len(required_pred_cols),
        "model_date_count": len(bundle["model_date_tags"]),
        "sleeve_component_count": len(variant["sleeve_components"]),
    })

v3_model_meta_df = pd.DataFrame(v3_meta_rows)
v3_export_manifest_df = pd.DataFrame(export_rows)
v3_meta_path = os.path.join(OUT_DIR, "model_csi800_lgb_v3_walkforward_model_meta.csv")
v3_manifest_path = os.path.join(OUT_DIR, "model_csi800_lgb_v3_walkforward_manifest.csv")
v3_model_meta_df.to_csv(v3_meta_path, index=False)
v3_export_manifest_df.to_csv(v3_manifest_path, index=False)

print("exported v3 walk-forward pkl files:")
print(v3_export_manifest_df.to_string(index=False))
print("model date count:", len(v3_model_bank), sorted(v3_model_bank.keys()))
print("required raw factor count:", len(all_required_factors))
print("model feature count:", len(all_model_features))
print("meta path:", v3_meta_path)
print("manifest path:", v3_manifest_path)



## 12. Yearly Walk-Forward Robustness Validation

验证 V3 在年度 expanding walk-forward 下的鲁棒性：2019-2022 测 2023，2019-2023 测 2024，2019-2024 测 2025，2019-2025 测 2026。


In [ ]:
# =========================
# V3 yearly walk-forward robustness validation
# =========================
V3_YEARLY_WF_OUT_DIR = "csi800_ml_mainline_family_v3_yearly_wf_outputs"
os.makedirs(V3_YEARLY_WF_OUT_DIR, exist_ok=True)

V3_YEARLY_WF_WINDOWS = [
    {"window_name": "train2019_2022_test2023", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"window_name": "train2019_2023_test2024", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"window_name": "train2019_2024_test2025", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"window_name": "train2019_2025_test2026", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

V3_YEARLY_KEEP_STRATEGIES = [
    ("raw_v22a_lgb_alpha_vs_alla", "top10"),
    ("raw_v22a_lgb_alpha_vs_alla", "top20"),
    ("v3_lgb_blend", "top20"),
    ("sleeve_60_base_top10_40_v3blend_top20", "sleeve"),
    ("sleeve_50_base_top20_50_v3blend_top20", "sleeve"),
    ("sleeve_60_base_top10_40_group_industry_top10", "sleeve"),
]


def slice_v3_yearly_window(all_df, spec):
    train_start = pd.Timestamp(spec["train_start"])
    train_end = pd.Timestamp(spec["train_end"])
    test_start = pd.Timestamp(spec["test_start"])
    test_end = pd.Timestamp(spec["test_end"])
    train_df = all_df[
        (all_df["rebalance_date"] >= train_start) &
        (all_df["rebalance_date"] <= train_end) &
        (all_df["next_date"] <= train_end)
    ].copy()
    test_df = all_df[
        (all_df["rebalance_date"] >= test_start) &
        (all_df["rebalance_date"] <= test_end)
    ].copy()
    return train_df, test_df


def add_v3_family_scores_for_window(train_df, valid_df, window_name):
    base_output_cols = [
        "stock", "rebalance_date", "next_date", "industry_bucket",
        "raw_return_1m", "benchmark_csi800_1m", "benchmark_alla_1m",
        "alpha_vs_csi800", "alpha_vs_alla", "manual_rule_score", "risk_penalty",
    ]
    train_score_df = train_df[base_output_cols].copy()
    score_df = valid_df[base_output_cols].copy()
    importance_parts = []
    model_meta = []

    base_train_pred, base_valid_pred, baseline_imp, baseline_meta = fit_predict_lgb_train_valid(train_df, valid_df, FACTOR_SET_V22A_37, "alpha_vs_alla", "baseline_lgb_score")
    train_score_df["baseline_lgb_score"] = base_train_pred
    score_df["baseline_lgb_score"] = base_valid_pred
    baseline_imp["model_family"] = "raw_v22a_lgb_alpha_vs_alla"
    baseline_meta["model_family"] = "raw_v22a_lgb_alpha_vs_alla"
    importance_parts.append(baseline_imp)
    model_meta.append(baseline_meta)

    hybrid_train_pred, hybrid_valid_pred, hybrid_imp, hybrid_meta = fit_predict_lgb_train_valid(train_df, valid_df, HYBRID_LIGHT_FEATURE_COLS, "alpha_vs_csi800", "hybrid_light_lgb_score")
    train_score_df["hybrid_light_lgb_score"] = hybrid_train_pred
    score_df["hybrid_light_lgb_score"] = hybrid_valid_pred
    hybrid_imp["model_family"] = "hybrid_light_lgb_alpha_vs_csi800"
    hybrid_meta["model_family"] = "hybrid_light_lgb_alpha_vs_csi800"
    importance_parts.append(hybrid_imp)
    model_meta.append(hybrid_meta)

    rank_csi_train_pred, rank_csi_valid_pred, rank_csi_imp, rank_csi_meta = fit_predict_lgb_train_valid(train_df, valid_df, RANK_INDUSTRY_FEATURE_COLS, "alpha_vs_csi800", "rank_industry_csi800_score")
    train_score_df["rank_industry_csi800_score"] = rank_csi_train_pred
    score_df["rank_industry_csi800_score"] = rank_csi_valid_pred
    rank_csi_imp["model_family"] = "rank_industry_lgb_alpha_vs_csi800"
    rank_csi_meta["model_family"] = "rank_industry_lgb_alpha_vs_csi800"
    importance_parts.append(rank_csi_imp)
    model_meta.append(rank_csi_meta)

    rank_alla_train_pred, rank_alla_valid_pred, rank_alla_imp, rank_alla_meta = fit_predict_lgb_train_valid(train_df, valid_df, RANK_INDUSTRY_FEATURE_COLS, "alpha_vs_alla", "rank_industry_alla_score")
    train_score_df["rank_industry_alla_score"] = rank_alla_train_pred
    score_df["rank_industry_alla_score"] = rank_alla_valid_pred
    rank_alla_imp["model_family"] = "rank_industry_lgb_alpha_vs_alla"
    rank_alla_meta["model_family"] = "rank_industry_lgb_alpha_vs_alla"
    importance_parts.append(rank_alla_imp)
    model_meta.append(rank_alla_meta)

    group_score_cols = []
    for group_name in ["value_cashflow", "quality_profit", "growth_balance", "momentum_risk", "technical_volume"]:
        cols = FACTOR_GROUPS[group_name]
        pred_col = "group_score_" + group_name
        train_pred, valid_pred, imp, meta = fit_predict_lgb_train_valid(train_df, valid_df, cols, "alpha_vs_alla", pred_col)
        train_score_df[pred_col] = train_pred
        score_df[pred_col] = valid_pred
        group_score_cols.append(pred_col)
        imp["model_family"] = "factor_group_recall_lgb"
        meta["model_family"] = "factor_group_recall_lgb"
        importance_parts.append(imp)
        model_meta.append(meta)

    for frame in [train_score_df, score_df]:
        frame["baseline_rank"] = rank_by_month(frame, "baseline_lgb_score").fillna(0.5)
        frame["hybrid_rank"] = rank_by_month(frame, "hybrid_light_lgb_score").fillna(0.5)
        frame["rank_industry_csi800_rank"] = rank_by_month(frame, "rank_industry_csi800_score").fillna(0.5)
        frame["rank_industry_alla_rank"] = rank_by_month(frame, "rank_industry_alla_score").fillna(0.5)
        frame["manual_rank"] = rank_by_month(frame, "manual_rule_score").fillna(0.5)
        frame["risk_rank"] = rank_by_month(frame, "risk_penalty").fillna(0.5)
        group_rank_cols = []
        for col in group_score_cols:
            rcol = col + "_rank"
            frame[rcol] = rank_by_month(frame, col).fillna(0.5)
            group_rank_cols.append(rcol)
        frame["group_mean_rank"] = frame[group_rank_cols].mean(axis=1)
        frame["factor_group_recall_lgb_score"] = 0.65 * frame["baseline_rank"] + 0.35 * frame["group_mean_rank"]
        frame["v3_lgb_blend_score"] = 0.55 * frame["baseline_rank"] + 0.30 * frame["manual_rank"] + 0.15 * frame["hybrid_rank"]

    rerank_recall_cols = [
        "baseline_lgb_score", "rank_industry_csi800_score", "rank_industry_alla_score",
        "factor_group_recall_lgb_score", "v3_lgb_blend_score", "manual_rule_score",
    ]
    rerank_feature_cols = unique_keep_order(
        RANK_INDUSTRY_FEATURE_COLS +
        ["baseline_rank", "hybrid_rank", "manual_rank", "risk_rank", "group_mean_rank", "rank_industry_csi800_rank", "rank_industry_alla_rank"] +
        group_score_cols
    )
    train_rerank_df = train_df.copy()
    valid_rerank_df = valid_df.copy()
    score_cols_to_merge = [c for c in train_score_df.columns if c not in base_output_cols]
    for c in score_cols_to_merge:
        train_rerank_df[c] = train_score_df[c].values
        valid_rerank_df[c] = score_df[c].values
    train_recall_mask = build_recall_mask(train_rerank_df, rerank_recall_cols, RERANK_RECALL_N)
    train_rerank_sample = train_rerank_df[train_recall_mask].copy()

    rerank_alla_train_pred, rerank_alla_valid_pred, rerank_alla_imp, rerank_alla_meta = fit_predict_lgb_train_valid(train_rerank_sample, valid_rerank_df, rerank_feature_cols, "alpha_vs_alla", "recall_rerank_alla_score")
    score_df["recall_rerank_alla_score"] = rerank_alla_valid_pred
    rerank_alla_imp["model_family"] = "recall_rerank_lgb_alpha_vs_alla"
    rerank_alla_meta["model_family"] = "recall_rerank_lgb_alpha_vs_alla"
    importance_parts.append(rerank_alla_imp)
    model_meta.append(rerank_alla_meta)

    rerank_csi_train_pred, rerank_csi_valid_pred, rerank_csi_imp, rerank_csi_meta = fit_predict_lgb_train_valid(train_rerank_sample, valid_rerank_df, rerank_feature_cols, "alpha_vs_csi800", "recall_rerank_csi800_score")
    score_df["recall_rerank_csi800_score"] = rerank_csi_valid_pred
    rerank_csi_imp["model_family"] = "recall_rerank_lgb_alpha_vs_csi800"
    rerank_csi_meta["model_family"] = "recall_rerank_lgb_alpha_vs_csi800"
    importance_parts.append(rerank_csi_imp)
    model_meta.append(rerank_csi_meta)

    importance_df = pd.concat(importance_parts, ignore_index=True)
    model_meta_df = pd.DataFrame(model_meta)
    importance_df["window_name"] = window_name
    model_meta_df["window_name"] = window_name
    score_df["window_name"] = window_name
    return score_df, importance_df, model_meta_df, group_score_cols


def evaluate_v3_yearly_window(score_df, group_score_cols, window_name):
    strategies = [
        ("manual_v3", "manual_rule_score", "manual_v3"),
        ("raw_v22a_lgb_alpha_vs_alla", "baseline_lgb_score", "baseline"),
        ("hybrid_light_lgb_alpha_vs_csi800", "hybrid_light_lgb_score", "hybrid_light"),
        ("rank_industry_lgb_alpha_vs_csi800", "rank_industry_csi800_score", "rank_industry"),
        ("rank_industry_lgb_alpha_vs_alla", "rank_industry_alla_score", "rank_industry"),
        ("factor_group_recall_lgb", "factor_group_recall_lgb_score", "factor_group_recall"),
        ("recall_rerank_lgb_alpha_vs_alla", "recall_rerank_alla_score", "recall_rerank"),
        ("recall_rerank_lgb_alpha_vs_csi800", "recall_rerank_csi800_score", "recall_rerank"),
        ("v3_lgb_blend", "v3_lgb_blend_score", "blend"),
    ]
    portfolio_profiles = ["top10", "top20", "industry_top10"]
    sleeve_defs = [
        ("sleeve_60_base_top10_40_v3blend_top20", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("v3_lgb_blend", "top20", 0.40)]),
        ("sleeve_60_base_top10_40_group_industry_top10", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("factor_group_recall_lgb", "industry_top10", 0.40)]),
        ("sleeve_60_base_top10_40_rankind_csi_top20", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("rank_industry_lgb_alpha_vs_csi800", "top20", 0.40)]),
        ("sleeve_60_base_top10_40_rerank_alla_top10", [("raw_v22a_lgb_alpha_vs_alla", "top10", 0.60), ("recall_rerank_lgb_alpha_vs_alla", "top10", 0.40)]),
        ("sleeve_50_base_top20_50_v3blend_top20", [("raw_v22a_lgb_alpha_vs_alla", "top20", 0.50), ("v3_lgb_blend", "top20", 0.50)]),
    ]

    summary_rows = []
    monthly_parts = []
    rank_ic_rows = []
    ret_lookup = {}
    for strategy_name, score_col, family_tag in strategies:
        ic_csi = monthly_rank_ic(score_df, score_col, "alpha_vs_csi800")
        ic_alla = monthly_rank_ic(score_df, score_col, "alpha_vs_alla")
        for portfolio_profile in portfolio_profiles:
            ret_df = monthly_portfolio_returns(score_df, strategy_name, score_col, portfolio_profile, group_score_cols)
            ret_df["window_name"] = window_name
            ret_lookup[(strategy_name, portfolio_profile)] = ret_df
            row = {
                "window_name": window_name,
                "strategy_name": strategy_name,
                "family_tag": family_tag,
                "score_col": score_col,
                "portfolio_profile": portfolio_profile,
                "rank_ic_csi800_mean": float(ic_csi.mean()) if len(ic_csi) else np.nan,
                "rank_ic_csi800_ir": float(ic_csi.mean() / ic_csi.std() * np.sqrt(12)) if len(ic_csi) and ic_csi.std() > 0 else np.nan,
                "rank_ic_alla_mean": float(ic_alla.mean()) if len(ic_alla) else np.nan,
                "rank_ic_alla_ir": float(ic_alla.mean() / ic_alla.std() * np.sqrt(12)) if len(ic_alla) and ic_alla.std() > 0 else np.nan,
                "avg_turnover": calc_turnover(ret_df),
                "drop_top1_excess": stress_without_top_months(ret_df, 1),
                "drop_top3_excess": stress_without_top_months(ret_df, 3),
            }
            row.update(summarize_monthly(ret_df))
            summary_rows.append(row)
            if not ret_df.empty:
                monthly_parts.append(ret_df)
        rank_ic_one = pd.DataFrame({
            "rebalance_date": ic_csi.index,
            "window_name": window_name,
            "strategy_name": strategy_name,
            "family_tag": family_tag,
            "score_col": score_col,
            "rank_ic_csi800": ic_csi.values,
            "rank_ic_alla": ic_alla.reindex(ic_csi.index).values,
        })
        rank_ic_rows.append(rank_ic_one)

    for sleeve_name, sleeve_parts in sleeve_defs:
        ret_df = combine_sleeve_returns(ret_lookup, sleeve_name, sleeve_parts)
        ret_df["window_name"] = window_name
        ret_lookup[(sleeve_name, "sleeve")] = ret_df
        row = {
            "window_name": window_name,
            "strategy_name": sleeve_name,
            "family_tag": "sleeve_portfolio",
            "score_col": "sleeve_weighted_components",
            "portfolio_profile": "sleeve",
            "rank_ic_csi800_mean": np.nan,
            "rank_ic_csi800_ir": np.nan,
            "rank_ic_alla_mean": np.nan,
            "rank_ic_alla_ir": np.nan,
            "avg_turnover": calc_turnover(ret_df),
            "drop_top1_excess": stress_without_top_months(ret_df, 1),
            "drop_top3_excess": stress_without_top_months(ret_df, 3),
        }
        row.update(summarize_monthly(ret_df))
        summary_rows.append(row)
        if not ret_df.empty:
            monthly_parts.append(ret_df)

    summary_df = pd.DataFrame(summary_rows)
    monthly_df = pd.concat(monthly_parts, ignore_index=True) if len(monthly_parts) else pd.DataFrame()
    rank_ic_df = pd.concat(rank_ic_rows, ignore_index=True) if len(rank_ic_rows) else pd.DataFrame()
    return summary_df, monthly_df, rank_ic_df


v3_ywf_score_parts = []
v3_ywf_importance_parts = []
v3_ywf_meta_parts = []
v3_ywf_summary_parts = []
v3_ywf_monthly_parts = []
v3_ywf_rank_ic_parts = []

for spec in V3_YEARLY_WF_WINDOWS:
    window_name = spec["window_name"]
    train_part, valid_part = slice_v3_yearly_window(df_all, spec)
    print("\n===== V3 yearly WF", window_name, "=====")
    print("train", train_part.shape, "months", train_part["rebalance_date"].nunique(), "valid", valid_part.shape, "months", valid_part["rebalance_date"].nunique())
    if len(train_part) < 10000 or valid_part.empty:
        print("skip insufficient window", window_name)
        continue
    score_part, imp_part, meta_part, group_cols = add_v3_family_scores_for_window(train_part, valid_part, window_name)
    summary_part, monthly_part, rank_ic_part = evaluate_v3_yearly_window(score_part, group_cols, window_name)
    v3_ywf_score_parts.append(score_part)
    v3_ywf_importance_parts.append(imp_part)
    v3_ywf_meta_parts.append(meta_part)
    v3_ywf_summary_parts.append(summary_part)
    v3_ywf_monthly_parts.append(monthly_part)
    v3_ywf_rank_ic_parts.append(rank_ic_part)
    del train_part, valid_part, score_part, imp_part, meta_part, summary_part, monthly_part, rank_ic_part
    gc.collect()

v3_ywf_scores_df = pd.concat(v3_ywf_score_parts, ignore_index=True, sort=False) if len(v3_ywf_score_parts) else pd.DataFrame()
v3_ywf_importance_df = pd.concat(v3_ywf_importance_parts, ignore_index=True, sort=False) if len(v3_ywf_importance_parts) else pd.DataFrame()
v3_ywf_model_meta_df = pd.concat(v3_ywf_meta_parts, ignore_index=True, sort=False) if len(v3_ywf_meta_parts) else pd.DataFrame()
v3_ywf_summary_df = pd.concat(v3_ywf_summary_parts, ignore_index=True, sort=False) if len(v3_ywf_summary_parts) else pd.DataFrame()
v3_ywf_monthly_df = pd.concat(v3_ywf_monthly_parts, ignore_index=True, sort=False) if len(v3_ywf_monthly_parts) else pd.DataFrame()
v3_ywf_rank_ic_df = pd.concat(v3_ywf_rank_ic_parts, ignore_index=True, sort=False) if len(v3_ywf_rank_ic_parts) else pd.DataFrame()

if not v3_ywf_summary_df.empty:
    v3_ywf_summary_df = v3_ywf_summary_df.sort_values(["window_name", "cum_excess_csi800"], ascending=[True, False])
    v3_ywf_mainline_summary_df = v3_ywf_summary_df[
        v3_ywf_summary_df.apply(lambda r: (r["strategy_name"], r["portfolio_profile"]) in V3_YEARLY_KEEP_STRATEGIES, axis=1)
    ].copy()
else:
    v3_ywf_mainline_summary_df = pd.DataFrame()

v3_ywf_scores_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_scores.csv"), index=False)
v3_ywf_importance_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_importance.csv"), index=False)
v3_ywf_model_meta_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_model_meta.csv"), index=False)
v3_ywf_summary_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_summary.csv"), index=False)
v3_ywf_mainline_summary_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_mainline_summary.csv"), index=False)
v3_ywf_monthly_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_monthly.csv"), index=False)
v3_ywf_rank_ic_df.to_csv(os.path.join(V3_YEARLY_WF_OUT_DIR, "v3_yearly_wf_rank_ic.csv"), index=False)

print("saved yearly WF outputs ->", V3_YEARLY_WF_OUT_DIR)
print("mainline summary:")
display(v3_ywf_mainline_summary_df)
